# Pandas from the Ground Up

A hands-on notebook for people who already know Python. No Python review here, only pandas.

**How to use this notebook**

1. Run the cells top to bottom (`Shift+Enter`). Later cells depend on earlier ones.
2. When you see **🔮 Predict**, stop and guess the output *before* running the cell. Being wrong and seeing why is the fastest way to build a mental model.
3. Each section ends with a **🧩 Exercise**. Write your answer in the empty cell, then open the collapsed solution to compare.
4. The final section is a capstone on the real `reviews.csv` file in this folder.

**Contents**

| # | Section | Key ideas |
|---|---|---|
| 1 | Why pandas? | What problem it solves, where it fits |
| 2 | `Series` | values + index, vectorization, **alignment** |
| 3 | `DataFrame` | a dict of Series sharing one index, inspecting data |
| 4 | Selecting data | `[]`, `.loc`, `.iloc`, boolean masks, `.query` |
| 5 | Creating & changing columns | `.assign`, `np.where`, `.map`, why to avoid `.apply` |
| 6 | Copy-on-Write | why `df[mask]["col"] = x` silently does nothing |
| 7 | Missing data | `NaN`, `isna`, `fillna`, `dropna`, nullable dtypes |
| 8 | dtypes & cleaning | `astype`, `to_numeric`, `.str`, `.dt`, `category` |
| 9 | Sorting & counting | `sort_values`, `nlargest`, `value_counts` |
| 10 | GroupBy | split-apply-combine, `agg` vs `transform` vs `filter` |
| 11 | Combining tables | `merge` (joins), `concat` |
| 12 | Reshaping | `pivot_table`, `melt`, `crosstab` |
| 13 | Time series | `resample`, `rolling`, `shift`, `pct_change` |
| 14 | Reading & writing files | `read_csv` options, chunking, Parquet |
| 15 | Method chaining | writing readable pipelines |
| 16 | Performance & pitfalls | vectorize, `iterrows`, memory |
| 17 | pandas vs other tools | Polars, DuckDB/SQL, Spark, Excel |
| 18 | Capstone | analysing `reviews.csv` |

---
## 1. Why pandas?

### The problem it solves
Most real-world data is **tabular**: rows of records, columns of fields (CSV exports, SQL tables, spreadsheets, logs, API responses). Plain Python handles this badly:

* A `list[dict]` is flexible but slow, and every operation ("average price per region") becomes a hand-written loop.
* NumPy arrays are fast, but they hold **one dtype** and have no column names or row labels.

pandas sits in between. It gives you **labelled, column-oriented tables**. Each column is a fast NumPy-style array with its own dtype, and there is a large API for the everyday jobs: load, clean, filter, join, group, reshape, and export.

### Why it became the default
* **Timing and maturity.** Wes McKinney started it in 2008 at a hedge fund to do financial analysis in Python instead of R or Excel. It was there when Python data science took off, and it grew up with it.
* **The ecosystem speaks DataFrame.** matplotlib, seaborn, scikit-learn, statsmodels, plotly, Streamlit, and most cloud and database SDKs accept or return DataFrames. Learning pandas unlocks all of them.
* **I/O breadth.** One-liners read and write CSV, Excel, JSON, SQL, Parquet, HTML tables, the clipboard, and more.
* **Notebook-friendly.** DataFrames render as tables, which makes exploratory analysis fast and visual.
* **Expressiveness.** "Average order value per region per month" is one readable line.
* **Community.** Almost every error you hit has a Stack Overflow answer, and LLMs know it well.

We come back to its **disadvantages** and to **alternatives** (Polars, DuckDB, Spark) in Section 17, once you have the vocabulary to judge them.

> **Version note.** This notebook targets **pandas 3.x** (this folder's `.venv` has 3.1.1). pandas 3 changed some old behaviour: Copy-on-Write is always on, strings get a dedicated `str` dtype, and some old methods were removed. Tutorials written before 2025 often show patterns that no longer work. The notebook flags these where they matter.

In [1]:
import numpy as np
import pandas as pd

print("pandas", pd.__version__)
print("numpy ", np.__version__)

# Optional display settings: show more columns and limit float noise
pd.set_option("display.max_columns", 20)
pd.set_option("display.precision", 2)

pandas 3.1.1
numpy  2.4.6


---
## 2. `Series`: the 1-D building block

A `Series` is **an array of values plus an index of labels**. Everything else in pandas builds on that pairing.

```
 index   values
┌──────┬────────┐
│ 'a'  │  10    │
│ 'b'  │  20    │      dtype: int64
│ 'c'  │  30    │      name:  'sales'
└──────┴────────┘
```

If you don't give an index, pandas uses `0, 1, 2, ...` (a `RangeIndex`).

In [2]:
s = pd.Series([10, 20, 30], index=["a", "b", "c"], name="sales")
s

a    10
b    20
c    30
Name: sales, dtype: int64

In [ ]:
print(s.values)   # the underlying NumPy array
print(s.index)    # the labels
print(s.dtype)    # one dtype for the whole Series
print(s["b"])     # access by label, like a dict

[10 20 30]
Index(['a', 'b', 'c'], dtype='str')
int64
20
20


### Vectorization: operate on the whole column at once
Arithmetic and comparisons apply to every element, without a Python loop. Under the hood this runs as compiled C over a contiguous array, which is typically **10–100× faster** than a `for` loop.

In [ ]:
prices = pd.Series([4.5, 12.0, 7.25, 30.0])

print(prices * 1.2)        # element-wise
print(prices > 10)         # element-wise comparison -> boolean Series
print(prices[prices > 10]) # boolean Series used as a filter ("mask")
print(prices.mean(), prices.max(), prices.sum())

0     5.4
1    14.4
2     8.7
3    36.0
dtype: float64
0    False
1     True
2    False
3     True
dtype: bool
1    12.0
3    30.0
dtype: float64
13.4375 30.0 53.75


### Index alignment, the idea that surprises everyone
When you combine two Series, pandas **matches rows by label, not by position**. Labels found in only one Series produce `NaN` (missing).

🔮 **Predict:** what is `jan + feb` below? Write down the value for each of `a`, `b`, `c`, `d`.

In [5]:
jan = pd.Series({"a": 1, "b": 2, "c": 3})
feb = pd.Series({"b": 10, "c": 20, "d": 30})

jan + feb

a     NaN
b    12.0
c    23.0
d     NaN
dtype: float64

`a` and `d` exist in only one Series, so the result is `NaN`. The dtype also became `float64`, because classic NumPy integers can't hold `NaN` (more on this in Section 7).

Alignment is a feature: it stops you from silently adding values that belong to different rows. It is also the source of many "where did these NaNs come from?" bugs. To treat missing labels as 0, use the method form:

In [ ]:
jan.add(feb, fill_value=0)

### 🧩 Exercise: Series basics

1. Build a Series `temps` of the temperatures `[22, 25, 19, 30, 27]`, indexed by `["Mon", "Tue", "Wed", "Thu", "Fri"]`.
2. Convert it to Fahrenheit (`F = C * 9/5 + 32`) without a loop.
3. Select only the days hotter than 24 °C.
4. 🔮 Predict, then check: what does `temps + pd.Series({"Mon": 1, "Sat": 5})` return?

<details>
<summary><b>Show solution</b> (try first!)</summary>

```python
temps = pd.Series([22, 25, 19, 30, 27], index=["Mon", "Tue", "Wed", "Thu", "Fri"])
print(temps * 9 / 5 + 32)
print(temps[temps > 24])
print(temps + pd.Series({"Mon": 1, "Sat": 5}))
# Only Mon is in both, so it becomes 23. Every other label (Tue-Fri, Sat) is NaN.
```
</details>

---
## 3. `DataFrame`: a table of Series that share one index

The most useful mental model: **a DataFrame is a dict of column name → Series, where all the Series share the same row index.**

```
            columns ──►  'product'   'price'   'qty'
 index  ┌──────────────┬───────────┬─────────┬───────┐
   │    │      0       │  'apple'  │  0.50   │  12   │
   ▼    │      1       │  'pear'   │  0.75   │   3   │
        │      2       │  'kiwi'   │  0.30   │  40   │
        └──────────────┴───────────┴─────────┴───────┘
                         dtype: str  float64   int64     ← one dtype PER COLUMN
```

So `df["price"]` gives you back a **Series**, and anything you can do to a Series, you can do to a column.

In [6]:
df = pd.DataFrame({
    "product": ["apple", "pear", "kiwi"],
    "price":   [0.50, 0.75, 0.30],
    "qty":     [12, 3, 40],
})
df

,product,price,qty
0,apple,0.50,12
1,pear,0.75,3
2,kiwi,0.30,40


In [7]:
print(type(df["price"]))   # a column is a Series
print(df.dtypes)           # one dtype per column
print(df.shape)            # (rows, columns)

<class 'pandas.Series'>
product        str
price      float64
qty          int64
dtype: object
(3, 3)


Other common ways to build a DataFrame:

In [8]:
# From a list of dicts (typical for JSON / API responses)
records = [
    {"id": 1, "city": "Toronto"}, 
    {"id": 2, "city": "Austin", "zip": "73301"}
]
print(pd.DataFrame(records), end="\n\n")   # missing keys become NaN

# From a 2-D NumPy array
print(pd.DataFrame(np.arange(6).reshape(2, 3), columns=["x", "y", "z"]))

   id     city    zip
0   1  Toronto    NaN
1   2   Austin  73301

   x  y  z
0  0  1  2
1  3  4  5


### The practice dataset
For the rest of the notebook we use a small, made-up **online store**: `customers`, `products` and `orders`. It's generated with a fixed random seed, so you get the same numbers every run. Don't worry about the generation code. It deliberately includes some **messiness** (missing quantities, orders from unknown customers) for us to deal with later.

In [10]:
rng = np.random.default_rng(42)

customers = pd.DataFrame({
    "customer_id": range(1, 101),
    "name": [f"Customer {i}" for i in range(1, 101)],
    "region": rng.choice(["North", "South", "East", "West"], size=100),
    "signup_date": pd.Timestamp("2025-01-01")
                   + pd.to_timedelta(rng.integers(0, 365, size=100), unit="D"),
})

products = pd.DataFrame({
    "product_id": ["P01", "P02", "P03", "P04", "P05", "P06", "P07", "P08"],
    "product":    ["Laptop", "Mouse", "Keyboard", "Monitor", "Desk", "Chair", "Lamp", "Webcam"],
    "category":   ["Electronics", "Electronics", "Electronics", "Electronics",
                   "Furniture", "Furniture", "Furniture", "Electronics"],
    "unit_price": [1200.0, 25.0, 75.0, 300.0, 450.0, 250.0, 40.0, 90.0],
})

n = 1_000
quantity = rng.integers(1, 6, size=n).astype(float)
quantity[rng.choice(n, size=25, replace=False)] = np.nan          # 25 missing quantities

orders = pd.DataFrame({
    "order_id":    range(1001, 1001 + n),
    "order_date":  pd.Timestamp("2026-01-01")
                   + pd.to_timedelta(rng.integers(0, 300, size=n), unit="D"),
    "customer_id": rng.integers(1, 106, size=n),                   # ids 101-105 don't exist!
    "product_id":  rng.choice(products["product_id"], size=n),
    "quantity":    quantity,
    "discount":    rng.choice([0.0, 0.0, 0.0, 0.1, 0.2], size=n),
})

orders.head()

,order_id,order_date,customer_id,product_id,quantity,discount
0,1001,2026-10-21,45,P04,2.0,0.0
1,1002,2026-03-14,54,P03,5.0,0.0
2,1003,2026-05-04,58,P06,3.0,0.0
3,1004,2026-01-23,26,P02,4.0,0.2
4,1005,2026-08-27,53,P03,3.0,0.0


### First look at any new dataset
Run these six things every time you meet a new DataFrame. Together they answer: how big is it, what types are the columns, what's missing, and what do the values look like?

In [11]:
orders.shape

(1000, 6)

In [12]:
orders.info()   # column dtypes, non-null counts, memory use

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     1000 non-null   int64         
 1   order_date   1000 non-null   datetime64[us]
 2   customer_id  1000 non-null   int64         
 3   product_id   1000 non-null   str           
 4   quantity     975 non-null    float64       
 5   discount     1000 non-null   float64       
dtypes: datetime64[us](1), float64(2), int64(2), str(1)
memory usage: 47.0 KB


In [13]:
orders.describe()   # summary stats for numeric (and datetime) columns

,order_id,order_date,customer_id,quantity,discount
count,1000.00,1000,1000.00,975.00,1000.00
mean,1500.50,2026-05-29 05:09:36,52.95,3.03,0.06
min,1001.00,2026-01-01 00:00:00,1.00,1.00,0.00
25%,1250.75,2026-03-16 00:00:00,25.75,2.00,0.00
50%,1500.50,2026-05-24 12:00:00,51.50,3.00,0.00
75%,1750.25,2026-08-12 00:00:00,80.00,4.00,0.10
max,2000.00,2026-10-27 00:00:00,105.00,5.00,0.20
std,288.82,NaN,30.45,1.43,0.08


In [14]:
print(orders.isna().sum(), end="\n\n")   # missing values per column
print(orders.nunique())                    # distinct values per column

order_id        0
order_date      0
customer_id     0
product_id      0
quantity       25
discount        0
dtype: int64

order_id       1000
order_date      293
customer_id     105
product_id        8
quantity          5
discount          3
dtype: int64


In [ ]:
orders.sample(5, random_state=0)   # random rows; often more honest than .head()

`info()` already told us something useful: `quantity` has **975 non-null** values out of 1000, so 25 are missing. We handle those in Section 7.

### 🧩 Exercise: Inspect the customers table

Using `customers`:
1. How many rows and columns does it have?
2. What is the dtype of `signup_date`?
3. How many customers are in each region? (Hint: there's a method on Series that counts distinct values. Try guessing its name, or look ahead to Section 9.)

In [ ]:
# your code here

<details>
<summary><b>Show solution</b> (try first!)</summary>

```python
print(customers.shape)
print(customers["signup_date"].dtype)
print(customers["region"].value_counts())
```
</details>

---
## 4. Selecting data

There are three main tools. Mixing them up is the most common beginner confusion, so it's worth learning them precisely.

| Syntax | Selects by | Typical use |
|---|---|---|
| `df["col"]` / `df[["a", "b"]]` | column **name** | grab one or more columns |
| `df[bool_mask]` | boolean Series | filter rows |
| `df.loc[rows, cols]` | **labels** (index values, column names) | the general-purpose, explicit tool |
| `df.iloc[rows, cols]` | integer **positions** (0, 1, 2…) | "first 5 rows", "last column" |

**Rule of thumb:** use `.loc` whenever you select rows *and* columns together, or whenever you assign.

In [26]:
orders["quantity"].head(3)                    # one column  -> Series

0    2.0
1    5.0
2    3.0
Name: quantity, dtype: float64

In [27]:
orders[["order_id", "quantity"]].head(3)      # list of columns -> DataFrame (note the double brackets)

,order_id,quantity
0,1001,2.0
1,1002,5.0
2,1003,3.0


### `.loc` vs `.iloc`
To make the difference visible, let's give `orders` a meaningful index: the `order_id`.

In [28]:
by_id = orders.set_index("order_id")
by_id.head(3)

,order_date,customer_id,product_id,quantity,discount
order_id,,,,,
1001,2026-10-21,45,P04,2.0,0.0
1002,2026-03-14,54,P03,5.0,0.0
1003,2026-05-04,58,P06,3.0,0.0


In [31]:
print(by_id.loc[1003], end="\n\n")                     # row whose LABEL is 1003
print(by_id.iloc[2], end="\n\n")                       # row at POSITION 2 (it's also 1003 here)
print(by_id.loc[1001:1003, ["product_id", "quantity"]])  # label slices INCLUDE the end
print(by_id.iloc[1:2])  # label slices INCLUDE the end

order_date     2026-05-04 00:00:00
customer_id                     58
product_id                     P06
quantity                       3.0
discount                       0.0
Name: 1003, dtype: object

order_date     2026-05-04 00:00:00
customer_id                     58
product_id                     P06
quantity                       3.0
discount                       0.0
Name: 1003, dtype: object

         product_id  quantity
order_id                     
1001            P04       2.0
1002            P03       5.0
1003            P06       3.0
         order_date  customer_id product_id  quantity  discount
order_id                                                       
1002     2026-03-14           54        P03       5.0       0.0


🔮 **Predict:** `by_id.loc[1001:1003]` returned 3 rows. How many rows will `by_id.iloc[0:3]` return? How about `by_id.iloc[0:2]`?

> **Key difference:** `.loc` slices are **inclusive** of the end label. `.iloc` slices follow normal Python rules and **exclude** the end.

In [ ]:
print(len(by_id.iloc[0:3]), len(by_id.iloc[0:2]))

### Boolean masks: filtering rows
A comparison on a column gives a boolean Series. Passing that Series back into `[]` or `.loc` keeps the rows where it's `True`.

Combine conditions with `&` (and), `|` (or) and `~` (not), **not** with `and`/`or`/`not`, and **wrap each condition in parentheses**. Python's operator precedence makes `&` bind tighter than `>`, so the parentheses are required.

In [ ]:
mask = orders["quantity"] >= 4
print(mask.head())
print("rows matching:", mask.sum())   # True counts as 1, so .sum() counts matches

In [ ]:
big_discounted = orders[(orders["quantity"] >= 4) & (orders["discount"] > 0)]
big_discounted.head()

In [ ]:
# Handy helpers that produce masks
orders[orders["product_id"].isin(["P01", "P04"])].head(3)        # membership

In [ ]:
orders[orders["order_date"].between("2026-03-01", "2026-03-07")].head(3)  # inclusive range

In [ ]:
# .loc with a mask AND a column selection at the same time
orders.loc[orders["discount"] == 0.2, ["order_id", "discount"]].head(3)

### `.query()`: SQL-like string filters
Same result, often more readable for complex conditions. Use `@name` to reference a Python variable.

In [ ]:
min_qty = 4
orders.query("quantity >= @min_qty and discount > 0 and product_id in ['P01', 'P04']").head()

### 🧩 Exercise: Selection

1. Using `.iloc`, select the **last 3 rows** and the **first 2 columns** of `orders`.
2. Select all orders for product `"P05"` with quantity exactly 5, showing only `order_id`, `order_date` and `quantity`.
3. How many orders are **not** in January 2026? (Hint: `~`, plus `.dt.month` for the month number of a datetime column.)

In [ ]:
# your code here

<details>
<summary><b>Show solution</b> (try first!)</summary>

```python
print(orders.iloc[-3:, :2])
print(orders.loc[(orders["product_id"] == "P05") & (orders["quantity"] == 5),
                 ["order_id", "order_date", "quantity"]])
print((~(orders["order_date"].dt.month == 1)).sum())
# or: (orders["order_date"].dt.month != 1).sum()
```
</details>

---
## 5. Creating and changing columns

Let's attach prices to the orders (we cover `merge` properly in Section 11) and compute revenue.

In [ ]:
orders = orders.merge(products[["product_id", "unit_price"]], on="product_id", how="left")

# Plain assignment: vectorized arithmetic between columns
orders["revenue"] = orders["quantity"] * orders["unit_price"] * (1 - orders["discount"])
orders.head()

### `.assign()`: create columns and return a NEW DataFrame
`df["x"] = ...` changes `df` in place. `df.assign(x=...)` returns a new DataFrame and leaves the original alone. That makes it the right tool for **method chains** (Section 15). Inside `assign`, a `lambda` receives the DataFrame *as it is at that point in the chain*.

In [ ]:
orders.assign(
    is_large   = lambda d: d["quantity"] >= 4,
    order_month = lambda d: d["order_date"].dt.to_period("M"),
).head()

### Conditional values
| Need | Tool |
|---|---|
| two outcomes | `np.where(cond, if_true, if_false)` |
| several outcomes | `np.select([cond1, cond2, ...], [val1, val2, ...], default=...)` |
| numeric bins | `pd.cut(values, bins, labels)` |
| replace by lookup | `Series.map(dict)` |

In [ ]:
tier = np.select(
    [orders["revenue"] >= 2000, orders["revenue"] >= 300],
    ["high", "medium"],
    default="low",
)
orders["tier"] = tier

orders["size"] = pd.cut(orders["quantity"], bins=[0, 1, 3, 5], labels=["single", "few", "many"])

discount_label = {0.0: "none", 0.1: "10% off", 0.2: "20% off"}
orders["discount_label"] = orders["discount"].map(discount_label)

orders[["revenue", "tier", "quantity", "size", "discount", "discount_label"]].head()

### What about `.apply()`?
`.apply(func)` calls a Python function **once per element (or per row)**. It's flexible, but it gives up vectorization: it's a Python loop in disguise.

**Use it only when no vectorized method exists.** Let's measure the difference:

In [ ]:
import time

t0 = time.perf_counter()
slow = orders.apply(lambda row: row["quantity"] * row["unit_price"] * (1 - row["discount"]), axis=1)
t1 = time.perf_counter()
fast = orders["quantity"] * orders["unit_price"] * (1 - orders["discount"])
t2 = time.perf_counter()

print(f"apply(axis=1): {(t1 - t0) * 1000:8.2f} ms")
print(f"vectorized:    {(t2 - t1) * 1000:8.2f} ms")
print("same result:", slow.equals(fast))

With only 1,000 rows the gap is already large, and it grows linearly. At a million rows it's the difference between milliseconds and minutes.

### Renaming and dropping

In [ ]:
tmp = orders.rename(columns={"unit_price": "price"}).drop(columns=["tier", "size", "discount_label"])
tmp.columns.tolist()

### 🧩 Exercise: New columns

1. Add a column `net_unit_price` to `orders` = `unit_price * (1 - discount)`.
2. Add a boolean column `weekend` that is `True` when the order was placed on a Saturday or Sunday. (Hint: `.dt.day_of_week` gives 0 = Monday … 6 = Sunday.)
3. Using `np.where`, add `shipping` = 0 if `revenue >= 100`, otherwise 9.99.

In [ ]:
# your code here

<details>
<summary><b>Show solution</b> (try first!)</summary>

```python
orders["net_unit_price"] = orders["unit_price"] * (1 - orders["discount"])
orders["weekend"] = orders["order_date"].dt.day_of_week >= 5
orders["shipping"] = np.where(orders["revenue"] >= 100, 0.0, 9.99)
orders[["unit_price", "discount", "net_unit_price", "order_date", "weekend", "revenue", "shipping"]].head()
```
</details>

🔮 **Before moving on:** some rows have `quantity = NaN`, so their `revenue` is `NaN`. What does `np.where(revenue >= 100, 0, 9.99)` give for those rows, and why? (Answer: `NaN >= 100` is `False`, so they get **9.99**. Missing data quietly flows into the "else" branch. Section 7 covers how to deal with it explicitly.)

---
## 6. Copy-on-Write: why chained assignment doesn't work

In pandas 3, **every indexing result behaves like an independent copy**. So this classic pattern does **nothing** to `orders`:

```python
orders[orders["discount"] > 0]["revenue"] = 0     # ❌ chained assignment
```

It's two steps: `orders[mask]` makes a new object, then `["revenue"] = 0` changes *that* object, which is thrown away immediately. pandas warns you with a `ChainedAssignmentError`.

**The fix:** do the row and column selection in **one** `.loc` call on the original DataFrame.

In [ ]:
demo = orders[["order_id", "discount", "revenue"]].head(5).copy()

# ❌ Wrong: modifies a temporary copy (pandas emits a warning)
demo[demo["discount"] > 0]["revenue"] = 0
print(demo, end="\n\n")        # unchanged!

# ✅ Right: one .loc call with row mask + column
demo.loc[demo["discount"] > 0, "revenue"] = 0
print(demo)

Related pandas 3 rules:
* If you take a subset and plan to modify it on purpose, take an explicit **`.copy()`**, as above. It makes your intent obvious.
* Avoid `inplace=True`. It's not faster and doesn't chain. Prefer `df = df.dropna()`.
* You'll still see `SettingWithCopyWarning` in old tutorials. That warning is gone in pandas 3 because Copy-on-Write made it unnecessary.

---
## 7. Missing data

pandas marks missing values as `NaN` (for floats), `NaT` (for datetimes) or `pd.NA` (for nullable dtypes). Every reduction (`sum`, `mean`, …) **skips missing values by default**.

In [ ]:
print(orders["quantity"].isna().sum(), "missing quantities")
print("mean skips NaN:", orders["quantity"].mean())
print("count of non-missing:", orders["quantity"].count(), "vs len:", len(orders))
orders

Your three options are: **drop** the rows, **fill** the gaps, or **keep** them and deal with them later. The right choice depends on what the missing value *means*. It's a business decision, not a pandas decision.

In [25]:
print(len(orders.dropna(subset=["quantity"])), "rows after dropping missing quantity")

# Fill with a constant, a statistic, or per-column values
filled_const  = orders["quantity"].fillna(1)
filled_median = orders["quantity"].fillna(orders["quantity"].median())
print(filled_const.isna().sum(), filled_median.isna().sum())

975 rows after dropping missing quantity
0 0


In [ ]:
# Forward/backward fill: carry the last known value forward (common in time series)
readings = pd.Series([1.0, np.nan, np.nan, 4.0, np.nan])
pd.DataFrame({"raw": readings, "ffill": readings.ffill(), "bfill": readings.bfill(),
              "interp": readings.interpolate()})

### Integers and missing values: nullable dtypes
`quantity` is conceptually an integer, but it's stored as `float64` because classic NumPy ints can't hold `NaN`. pandas has **nullable** dtypes (capitalised names: `Int64`, `Float64`, `boolean`) that hold real integers plus `pd.NA`.

In [ ]:
q = orders["quantity"].astype("Int64")
print(q.dtype)
print(q.head(8).tolist())

### 🧩 Exercise: Missing data

The store's policy: an order with a missing quantity should be treated as **quantity 1**.
1. Fill the missing `quantity` values in `orders` with 1 and store the column as nullable `Int64`.
2. Recompute `revenue` from the cleaned quantity.
3. Confirm there are no missing values left in `quantity` or `revenue`.

In [ ]:
# your code here

<details>
<summary><b>Show solution</b> (try first!)</summary>

```python
orders["quantity"] = orders["quantity"].fillna(1).astype("Int64")
orders["revenue"] = orders["quantity"] * orders["unit_price"] * (1 - orders["discount"])
print(orders[["quantity", "revenue"]].isna().sum())
```
</details>

---
## 8. dtypes and cleaning messy columns

Real files arrive messy: numbers stored as text with currency symbols, dates as strings, inconsistent capitalisation. Getting each column into the **right dtype** is usually the first real job, because the dtype decides what you can do with the column.

| dtype | Holds | Notes |
|---|---|---|
| `int64`, `float64` | numbers | fast NumPy types |
| `Int64`, `Float64`, `boolean` | numbers/bools **with** `pd.NA` | nullable |
| `str` | text | pandas 3's default string dtype (was `object`) |
| `datetime64[ns]` | timestamps | unlocks `.dt` |
| `category` | small set of repeated values | saves memory, can be ordered |
| `object` | anything Python | slow fallback; usually a sign something needs cleaning |

In [ ]:
import io

raw_csv = """id,price,date,city,status
1,"$1,200.00",2026-01-05, toronto ,Shipped
2,$35.50,2026-01-07,Toronto,shipped
3,N/A,2026-02-11,AUSTIN,Pending
4,$99,not a date,austin ,SHIPPED
"""

messy = pd.read_csv(io.StringIO(raw_csv))   # StringIO lets us treat a string as a file
print(messy.dtypes)
messy

Everything is text. Let's fix each column with the right accessor:
* **`.str`** gives vectorized string methods (`strip`, `lower`, `replace`, `contains`, `split`, …).
* **`pd.to_numeric(..., errors="coerce")`** converts, and turns anything unparseable into `NaN` instead of crashing.
* **`pd.to_datetime(..., errors="coerce")`** does the same for dates (`NaT` for bad ones).

In [ ]:
clean = messy.assign(
    price  = lambda d: pd.to_numeric(d["price"].str.replace(r"[$,]", "", regex=True), errors="coerce"),
    date   = lambda d: pd.to_datetime(d["date"], errors="coerce"),
    city   = lambda d: d["city"].str.strip().str.title(),
    status = lambda d: d["status"].str.lower().astype("category"),
)
print(clean.dtypes)
clean

### `.dt`: datetime parts
Once a column is `datetime64`, the `.dt` accessor exposes its parts.

In [ ]:
d = orders["order_date"]
pd.DataFrame({
    "date": d, "year": d.dt.year, "month": d.dt.month, "day_name": d.dt.day_name(),
    "quarter": d.dt.quarter, "week": d.dt.isocalendar().week,
}).head()

### `category`: memory and meaning
A column with few distinct values repeated many times (region, status, country) is a good `category`. pandas stores each distinct value once, plus small integer codes. Ordered categories also sort and compare logically.

In [ ]:
regions_str = pd.Series(rng.choice(["North", "South", "East", "West"], size=100_000))
regions_cat = regions_str.astype("category")
print(f"str:      {regions_str.memory_usage(deep=True) / 1e6:.2f} MB")
print(f"category: {regions_cat.memory_usage(deep=True) / 1e6:.2f} MB")

sizes = pd.Categorical(["M", "S", "L", "M"], categories=["S", "M", "L"], ordered=True)
print(pd.Series(sizes).sort_values().tolist(), sizes > "S")   # sorts S < M < L, not alphabetically

### 🧩 Exercise: Clean a column

Given `phones = pd.Series([" (416) 555-0101", "416.555.0199 ", "4165550142", None])`:
1. Strip everything except digits (hint: `.str.replace(r"\D", "", regex=True)`).
2. Add a boolean Series that's `True` when the cleaned number has exactly 10 digits (`.str.len()`).
3. 🔮 What happens to the `None` value at each step?

In [ ]:
phones = pd.Series([" (416) 555-0101", "416.555.0199 ", "4165550142", None])
# your code here

<details>
<summary><b>Show solution</b> (try first!)</summary>

```python
phones = pd.Series([" (416) 555-0101", "416.555.0199 ", "4165550142", None])
digits = phones.str.replace(r"\D", "", regex=True)
print(digits)
print(digits.str.len() == 10)
# The missing value stays missing through the .str methods. The comparison
# then gives False for it, because a missing value is never "equal to 10".
```
</details>

---
## 9. Sorting, ranking and counting

These answer the most common first-pass questions: *what's the biggest?* and *how often does each value appear?*

In [ ]:
orders.sort_values("revenue", ascending=False).head(3)

In [ ]:
# Sort by several columns: date ascending, then revenue descending within each date
orders.sort_values(["order_date", "revenue"], ascending=[True, False]).head(3)

In [ ]:
orders.nlargest(3, "revenue")[["order_id", "product_id", "revenue"]]   # faster than sort + head

In [ ]:
print(orders["product_id"].value_counts(), end="\n\n")                # counts, sorted descending
print(orders["discount"].value_counts(normalize=True).round(3))        # proportions instead

In [ ]:
orders["revenue_rank"] = orders["revenue"].rank(ascending=False, method="min")
orders[["order_id", "revenue", "revenue_rank"]].sort_values("revenue_rank").head()

---
## 10. GroupBy: split, apply, combine

This is the most important section. Almost every "per X" question ("revenue **per** product", "average order **per** customer **per** month") is a `groupby`.

```
          SPLIT                    APPLY (sum)        COMBINE
 ┌─────┬─────┐
 │ P01 │ 1200│──┐  P01: [1200, 2400]  ──► 3600 ─┐
 │ P02 │   25│──┼► P02: [25, 50]      ──►   75 ─┼──►  P01  3600
 │ P01 │ 2400│──┘  ...                           │     P02    75
 │ P02 │   50│                                   ┘     ...
 └─────┴─────┘
```

The pattern is `df.groupby(KEY)[COLUMN(S)].AGGREGATION()`.

In [ ]:
orders.groupby("product_id")["revenue"].sum()

In [ ]:
# Group by several keys -> a MultiIndex result. .reset_index() turns it back into a flat table.
orders.groupby(["product_id", "discount"])["revenue"].sum().reset_index().head(6)

### Several aggregations at once: named aggregation
`.agg(new_name=(column, function))` is the clearest form. You control the output column names, and the code reads like a spec.

In [ ]:
product_summary = (
    orders.groupby("product_id")
          .agg(
              n_orders     = ("order_id", "count"),
              units        = ("quantity", "sum"),
              revenue      = ("revenue", "sum"),
              avg_discount = ("discount", "mean"),
              last_order   = ("order_date", "max"),
          )
          .sort_values("revenue", ascending=False)
)
product_summary

### `agg` vs `transform` vs `filter`: the shape of the output
This is the part that trips most people up.

| Method | Output shape | Use it to… |
|---|---|---|
| `.agg()` / `.sum()` etc. | **one row per group** | summarise |
| `.transform()` | **same shape as the input** (one value per original row) | attach a group statistic to every row ("% of product total", "difference from group mean") |
| `.filter()` | a **subset of the original rows** | keep or drop whole groups ("customers with ≥ 15 orders") |

🔮 **Predict:** `orders` has 1000 rows and 8 products. How many rows does each of the next two cells return?

In [ ]:
orders.groupby("product_id")["revenue"].sum().shape

In [ ]:
orders.groupby("product_id")["revenue"].transform("sum").shape

In [ ]:
# transform in action: each order's share of its product's total revenue
orders["share_of_product"] = orders["revenue"] / orders.groupby("product_id")["revenue"].transform("sum")
orders[["order_id", "product_id", "revenue", "share_of_product"]].head()

In [ ]:
# filter in action: keep only orders from customers who placed at least 15 orders
loyal = orders.groupby("customer_id").filter(lambda g: len(g) >= 15)
print(loyal["customer_id"].nunique(), "loyal customers,", len(loyal), "orders")

### 🧩 Exercise: GroupBy

1. Total revenue per month. (Hint: group by `orders["order_date"].dt.to_period("M")`. You can group by any Series, not only a column name.)
2. For each customer, find the number of orders, total revenue and average revenue per order. Show the top 5 by total revenue.
3. Add a column `vs_product_avg` to `orders`: each order's revenue minus the average revenue for its product. Should you use `agg` or `transform`?

In [ ]:
# your code here

<details>
<summary><b>Show solution</b> (try first!)</summary>

```python
# 1
print(orders.groupby(orders["order_date"].dt.to_period("M"))["revenue"].sum())

# 2
print(orders.groupby("customer_id")
            .agg(n_orders=("order_id", "count"),
                 total=("revenue", "sum"),
                 avg=("revenue", "mean"))
            .nlargest(5, "total"))

# 3: transform, because we need one value per ORIGINAL row
orders["vs_product_avg"] = orders["revenue"] - orders.groupby("product_id")["revenue"].transform("mean")
orders[["product_id", "revenue", "vs_product_avg"]].head()
```
</details>

---
## 11. Combining tables: `merge` and `concat`

### `merge` = SQL `JOIN`
`pd.merge(left, right, on=key, how=...)` matches rows on key columns.

| `how=` | Keeps | SQL |
|---|---|---|
| `"inner"` (default) | only keys found in **both** | `INNER JOIN` |
| `"left"` | every row of `left` (NaN where no match) | `LEFT JOIN` |
| `"right"` | every row of `right` | `RIGHT JOIN` |
| `"outer"` | everything from both | `FULL OUTER JOIN` |

Remember that our orders include `customer_id`s 101–105, which **don't exist** in `customers`. Watch what each join does with them.

In [ ]:
inner = orders.merge(customers, on="customer_id", how="inner")
left  = orders.merge(customers, on="customer_id", how="left")
print("orders:", len(orders), "| inner:", len(inner), "| left:", len(left))

In [ ]:
# indicator=True adds a _merge column that shows where each row came from: great for debugging joins
check = orders.merge(customers, on="customer_id", how="left", indicator=True)
print(check["_merge"].value_counts(), end="\n\n")
print("orphan customer ids:", sorted(check.loc[check["_merge"] == "left_only", "customer_id"].unique()))

### Protect yourself with `validate=`
A silent **many-to-many** join (duplicate keys on both sides) multiplies rows, and totals get quietly inflated. `validate` makes pandas raise an error if the key relationship isn't what you expect.

In [ ]:
# Each order has exactly one customer: many orders -> one customer
enriched = orders.merge(customers, on="customer_id", how="left", validate="many_to_one")

# What happens when the right side has a duplicate key?
dup_customers = pd.concat([customers, customers.head(1)])
try:
    orders.merge(dup_customers, on="customer_id", validate="many_to_one")
except pd.errors.MergeError as e:
    print("MergeError:", e)

Other useful `merge` options:
* `left_on="cust_id", right_on="customer_id"` when the key columns have different names.
* `suffixes=("_order", "_customer")` controls how clashing column names are renamed.

### `concat`: stacking tables
`concat` glues DataFrames together **without matching keys**. Use it to stack rows (e.g. monthly files with the same columns) or to put columns side by side.

In [ ]:
jan_df = pd.DataFrame({"month": ["Jan", "Jan"], "sales": [100, 120]})
feb_df = pd.DataFrame({"month": ["Feb"], "sales": [90], "returns": [3]})

pd.concat([jan_df, feb_df], ignore_index=True)   # columns are unioned; missing -> NaN

A very common real-world pattern is to read a folder of CSVs and stack them:
```python
from pathlib import Path
df = pd.concat((pd.read_csv(p) for p in Path("data").glob("*.csv")), ignore_index=True)
```

### 🧩 Exercise: Merge

1. Build a DataFrame `full` that has every order with its customer's `region` **and** the product's `category`. Keep orders even if the customer is unknown. Use `validate`.
2. Fill the missing `region` values with `"Unknown"`.
3. Total revenue by `region` and `category`.

In [ ]:
# your code here

<details>
<summary><b>Show solution</b> (try first!)</summary>

```python
full = (
    orders
    .merge(customers[["customer_id", "region"]], on="customer_id", how="left", validate="many_to_one")
    .merge(products[["product_id", "category"]], on="product_id", how="left", validate="many_to_one")
)
full["region"] = full["region"].fillna("Unknown")
full.groupby(["region", "category"])["revenue"].sum()
```
</details>

---
## 12. Reshaping: long vs wide

The same data can be laid out two ways:

* **Long ("tidy")**: one row per observation, e.g. `(region, category, revenue)`. Best for groupby, plotting libraries and databases.
* **Wide**: one row per entity, one column per variable value, e.g. regions as rows and categories as columns. Best for humans reading reports, and for spreadsheets.

`pivot_table` goes long → wide (with aggregation). `melt` goes wide → long.

In [ ]:
# Rebuild `full` here, in case you skipped the exercise above
full = (
    orders
    .merge(customers[["customer_id", "region"]], on="customer_id", how="left")
    .merge(products[["product_id", "category"]], on="product_id", how="left")
    .assign(region=lambda d: d["region"].fillna("Unknown"))
)

wide = full.pivot_table(index="region", columns="category", values="revenue",
                        aggfunc="sum", margins=True, margins_name="Total")
wide.round(0)

In [ ]:
# melt: wide -> long. Each (region, category) cell becomes a row again.
long = (wide.drop(index="Total", columns="Total")
            .reset_index()
            .melt(id_vars="region", var_name="category", value_name="revenue"))
long.head(6)

In [ ]:
# crosstab: a quick frequency table of two columns (counts by default)
pd.crosstab(full["region"], full["discount_label"], normalize="index").round(2)

> `pivot` (without `_table`) only reshapes and raises an error if there are duplicate index/column pairs. `pivot_table` aggregates duplicates. When in doubt, use `pivot_table`.
>
> `stack()` / `unstack()` do the same kind of reshaping by moving levels between the row index and the columns. `groupby([...]).sum().unstack()` is a common pivot shortcut.

### 🧩 Exercise: Reshape

1. Build a pivot table with **months** as rows (`order_date.dt.to_period("M")`), **categories** as columns, and the **number of orders** as values.
2. Which month had the most Furniture orders?

In [ ]:
# your code here

<details>
<summary><b>Show solution</b> (try first!)</summary>

```python
monthly = full.pivot_table(index=full["order_date"].dt.to_period("M"),
                           columns="category", values="order_id", aggfunc="count")
print(monthly)
print("Most furniture orders:", monthly["Furniture"].idxmax())
```
</details>

---
## 13. Time series

pandas came out of finance, so its time-series tools are strong. The key idea is to put a `DatetimeIndex` on the data (or pass `on="date_column"`), and then you can resample, roll and shift by **calendar time**.

In [ ]:
daily = (orders.groupby("order_date")["revenue"].sum()
               .asfreq("D", fill_value=0))       # make every calendar day present
daily.head()

In [ ]:
# resample = groupby over time buckets. "W" = weekly, "ME" = month-end, "QE" = quarter-end
weekly  = daily.resample("W").sum()
monthly = daily.resample("ME").agg(["sum", "mean", "max"])
monthly

In [ ]:
ts = pd.DataFrame({"revenue": daily})
ts["rolling_7d"]   = ts["revenue"].rolling(7).mean()   # 7-day moving average (smooths noise)
ts["prev_day"]     = ts["revenue"].shift(1)            # value from the previous row
ts["day_change"]   = ts["revenue"].diff()              # revenue - prev_day
ts["cumulative"]   = ts["revenue"].cumsum()            # running total
ts.iloc[5:12]

In [ ]:
# Month-over-month growth
monthly_rev = daily.resample("ME").sum()
(monthly_rev.pct_change() * 100).round(1).rename("MoM growth %")

In [ ]:
# Slicing a DatetimeIndex with partial date strings
daily.loc["2026-03"].sum(), daily.loc["2026-03-01":"2026-03-15"].sum()

Plotting is one call away. pandas wraps matplotlib, so `df.plot()` works directly (it needs `matplotlib` installed, which this folder's `.venv` doesn't have yet: `uv add matplotlib`).
```python
ts[["revenue", "rolling_7d"]].plot(figsize=(10, 4), title="Daily revenue")
```

### 🧩 Exercise: Time series

1. Using `daily`, compute **weekly** revenue and find the best week.
2. Compute a **4-week** rolling average of that weekly revenue.
3. On which weekday (Monday…Sunday) does the store earn the most, on average? (Hint: group `daily` by `daily.index.day_name()`.)

In [ ]:
# your code here

<details>
<summary><b>Show solution</b> (try first!)</summary>

```python
weekly = daily.resample("W").sum()
print("Best week ending:", weekly.idxmax().date(), "revenue:", round(weekly.max(), 2))
print(weekly.rolling(4).mean().tail())
print(daily.groupby(daily.index.day_name()).mean().sort_values(ascending=False))
```
</details>

---
## 14. Reading and writing data

| Format | Read | Write | Notes |
|---|---|---|---|
| CSV | `pd.read_csv` | `df.to_csv` | universal, but no types; slow for big files |
| Excel | `pd.read_excel` | `df.to_excel` | needs `openpyxl` |
| JSON | `pd.read_json` / `pd.json_normalize` | `df.to_json` | `json_normalize` flattens nested API responses |
| SQL | `pd.read_sql` | `df.to_sql` | needs SQLAlchemy or a DB-API connection |
| Parquet | `pd.read_parquet` | `df.to_parquet` | **columnar, typed, compressed**: the best format for analytics; needs `pyarrow` |

### `read_csv` options that save you time

In [ ]:
csv_text = orders.head(50).to_csv(index=False)   # pretend this is a file on disk

df_small = pd.read_csv(
    io.StringIO(csv_text),
    usecols=["order_id", "order_date", "product_id", "quantity", "revenue"],  # only load what you need
    dtype={"product_id": "category", "quantity": "Int64"},                    # set types up front
    parse_dates=["order_date"],                                               # parse dates on load
    index_col="order_id",
)
df_small.info()

Other options you'll reach for: `sep=";"`, `encoding="utf-8"`, `na_values=["N/A", "-"]`, `skiprows=`, `nrows=` (peek at a huge file), `thousands=","`, `on_bad_lines="warn"`.

### Files bigger than memory: `chunksize`
`chunksize` returns an iterator of DataFrames, so you can process a huge file a piece at a time and combine the partial results.

In [ ]:
totals = []
for chunk in pd.read_csv(io.StringIO(orders.to_csv(index=False)), chunksize=250):
    totals.append(chunk.groupby("product_id")["revenue"].sum())

pd.concat(totals).groupby(level=0).sum()   # combine the partial sums

### Nested JSON → flat table

In [ ]:
api_response = [
    {"id": 1, "user": {"name": "Ana", "address": {"city": "Lima"}}, "tags": ["new"]},
    {"id": 2, "user": {"name": "Ben", "address": {"city": "Oslo"}}, "tags": []},
]
pd.json_normalize(api_response)

Writing is symmetric. Usually you want `index=False` for CSV, so the row numbers don't become a column:
```python
df.to_csv("out.csv", index=False)
df.to_parquet("out.parquet")          # keeps dtypes; much smaller and faster than CSV
```

---
## 15. Method chaining: readable pipelines

Most pandas methods return a new DataFrame, so you can chain them, one transformation per line, wrapped in parentheses. Compare the two styles below. Both answer the same question:

> *For known customers, what is each region's average monthly revenue from discounted Electronics orders?*

In [ ]:
# Style 1: many intermediate variables (easy to lose track of which one is current)
t1 = orders.merge(customers, on="customer_id", how="inner")
t2 = t1.merge(products[["product_id", "category"]], on="product_id")
t3 = t2[(t2["category"] == "Electronics") & (t2["discount"] > 0)]
t3 = t3.copy()
t3["month"] = t3["order_date"].dt.to_period("M")
t4 = t3.groupby(["region", "month"])["revenue"].sum()
answer1 = t4.groupby("region").mean().sort_values(ascending=False)
answer1

In [ ]:
# Style 2: one chain that reads top to bottom like a recipe
answer2 = (
    orders
    .merge(customers, on="customer_id", how="inner", validate="many_to_one")
    .merge(products[["product_id", "category"]], on="product_id", validate="many_to_one")
    .query("category == 'Electronics' and discount > 0")
    .assign(month=lambda d: d["order_date"].dt.to_period("M"))
    .groupby(["region", "month"])["revenue"].sum()
    .groupby("region").mean()
    .sort_values(ascending=False)
)
print("same answer:", answer2.equals(answer1))

`.pipe(func)` lets you put **your own functions** into a chain. That's how you keep larger pipelines testable: each step is a small, named, pure function.

In [ ]:
def add_month(df):
    return df.assign(month=df["order_date"].dt.to_period("M"))

def only_known_customers(df, valid_ids):
    return df[df["customer_id"].isin(valid_ids)]

(orders
    .pipe(only_known_customers, valid_ids=customers["customer_id"])
    .pipe(add_month)
    .groupby("month")["revenue"].sum()
    .head(3))

**Debugging tip:** comment out the lines of a chain from the bottom up to inspect any intermediate step.

---
## 16. Performance and common pitfalls

### The speed hierarchy (fastest first)
1. **Built-in vectorized methods**: `df["a"] * df["b"]`, `.sum()`, `.str.lower()`, `groupby().agg("mean")`
2. **NumPy functions** on columns: `np.where`, `np.select`, `np.log`
3. **`.map(dict)`** for lookups
4. **`.apply(func)`** on a Series (a Python loop, element by element)
5. **`.apply(func, axis=1)`** / **`iterrows()`** (a Python loop, *building a Series for every row*)

If you catch yourself writing `for i, row in df.iterrows():`, stop and look for a vectorized version. If you really must loop, `itertuples()` is much faster than `iterrows()`.

In [ ]:
t0 = time.perf_counter()
total = 0.0
for _, row in orders.iterrows():
    total += row["revenue"]
t1 = time.perf_counter()
total_vec = orders["revenue"].sum()
t2 = time.perf_counter()
print(f"iterrows: {(t1 - t0) * 1000:7.2f} ms")
print(f".sum():   {(t2 - t1) * 1000:7.2f} ms")

### Pitfall checklist

| Pitfall | Symptom | Fix |
|---|---|---|
| Chained assignment `df[m]["c"] = v` | nothing changes, warning | `df.loc[m, "c"] = v` |
| `and` / `or` in masks | `ValueError: truth value of a Series is ambiguous` | `&`, `\|`, `~` with parentheses |
| Unexpected `NaN` after arithmetic | values don't line up | index alignment: check the indexes, or `reset_index(drop=True)` |
| Row count grew after `merge` | inflated totals | duplicate keys: use `validate=` |
| Numbers stored as text | `sum()` concatenates strings / `TypeError` | `pd.to_numeric(errors="coerce")` |
| `groupby` result has an odd index | can't select a column you grouped by | `.reset_index()` or `as_index=False` |
| Comparing to `NaN` | `x == np.nan` is always `False` | `.isna()` |
| `.loc` slicing off by one | an extra row | `.loc` includes the end label, `.iloc` doesn't |
| Running out of RAM | `MemoryError`, swapping | `usecols`, `dtype=`, `category`, `chunksize`, Parquet, or a different tool (Section 17) |

In [ ]:
# Check how much memory a DataFrame really uses (deep=True counts string contents)
print(f"{orders.memory_usage(deep=True).sum() / 1e6:.2f} MB")
orders.memory_usage(deep=True).sort_values(ascending=False).head()

---
## 17. pandas vs other tools

Now that you know how pandas works, its strengths and weaknesses make more sense: most of them follow from its design.

### Design decisions behind pandas
* **Eager execution:** every line runs immediately and produces a full intermediate result in memory.
* **Mostly single-threaded:** most operations use one CPU core.
* **In-memory:** the whole dataset (plus intermediates) has to fit in RAM.
* **Row index:** labels on rows give you alignment and time-series features, but add complexity.
* **NumPy heritage:** this explains the historical dtype quirks (ints can't hold NaN, strings as `object`). pandas 3 fixes many of them.

### Advantages
* **The ecosystem.** It's the lingua franca of Python data work: plotting, ML (scikit-learn), stats, dashboards, cloud SDKs.
* **Huge, flexible API.** Messy real-world cleaning, reshaping, time series, Excel/JSON/SQL I/O, all in one library.
* **Interactive and exploratory.** Ideal in notebooks: inspect, tweak, re-run.
* **Mature and stable.** 15+ years of production use, excellent docs, and answers to almost every question online.
* **Strong time-series tools** (resample, rolling, shift, time zones, business days).

### Disadvantages
* **Memory hungry.** A long-standing rule of thumb from pandas' creator is to have **5–10× the dataset's size in RAM**. A 2 GB CSV can need 10–20 GB.
* **Slower on large data** than modern engines, because of single-threaded, eager execution and no query optimiser.
* **Large, inconsistent API.** There are often 3+ ways to do the same thing, plus legacy behaviour (and old tutorials) that pandas 3 changed.
* **The index is a double-edged sword.** It powers alignment, and it also causes surprise NaNs and confusing MultiIndexes.
* **Not distributed.** One machine only.

### The alternatives

| Tool | What it is | Pick it over pandas when… | Trade-offs |
|---|---|---|---|
| **Polars** | DataFrame library written in Rust, on Apache Arrow | data is large-ish (GBs), speed matters, you're starting a new pipeline | multi-threaded, **lazy** query optimiser, often 5–20× faster with far less memory, stricter and more consistent expression API, **no index**. Smaller ecosystem; some libraries still expect pandas (though `.to_pandas()` is cheap). |
| **DuckDB** | in-process analytical SQL database | your team thinks in SQL, you're querying big CSV/Parquet files, or the data is bigger than RAM | very fast, can spill to disk, can **query pandas DataFrames directly** with SQL. SQL is less convenient for step-by-step, imperative cleaning. |
| **SQL in a database** (Postgres, Snowflake, BigQuery) | the data already lives in a warehouse | the data is huge and already in the DB: push the work to where the data is | you need a database and a connection; awkward for iterative exploration and ML prep. Common pattern: aggregate in SQL, then finish in pandas. |
| **PySpark / Dask** | distributed / parallel DataFrames | data doesn't fit on one machine (100s of GB to TB), or you're on a Spark platform like Databricks | cluster overhead and latency; overkill for small data. Spark also offers a pandas-like API (`pyspark.pandas`). |
| **NumPy** | n-dimensional arrays | homogeneous numeric data: matrices, images, simulations | no column names, mixed types or groupby. pandas is built on it. |
| **Excel / Google Sheets** | spreadsheets | one-off, small, non-programmer audience | not reproducible, error-prone, ~1M row limit, hard to automate. pandas reads and writes Excel, so they work well together. |

### A practical decision guide
* **Under ~1 GB, exploratory, or feeding plots/ML** → **pandas**. It's the default for a reason.
* **Several GB on a laptop, or a performance-critical pipeline** → **Polars** or **DuckDB**.
* **Data already in a warehouse** → **SQL first**, then pandas for the last mile.
* **Bigger than one machine** → **Spark** (or Dask).

These tools are increasingly **interoperable through Apache Arrow**, so mixing them is normal: e.g. DuckDB for a heavy join over Parquet files, then `.df()` into pandas for plotting.

In [ ]:
# For a taste of the alternatives (only if you install them: `uv add polars duckdb`):
#
# import polars as pl
# (pl.from_pandas(orders)
#    .lazy()
#    .group_by("product_id")
#    .agg(pl.col("revenue").sum())
#    .sort("revenue", descending=True)
#    .collect())
#
# import duckdb
# duckdb.sql("SELECT product_id, SUM(revenue) AS revenue FROM orders GROUP BY 1 ORDER BY 2 DESC").df()
#
# Notice that the concepts (select, filter, group, aggregate, join) carry over exactly.
# What you learned here transfers; only the syntax changes.

### 🧠 Reflect
Before the capstone, answer these in your own words (in a markdown cell or on paper):
1. Why does `df[df.a > 0]["b"] = 1` not change `df`?
2. When would you use `transform` rather than `agg`?
3. A colleague has a 30 GB set of Parquet files and a 16 GB laptop. What would you suggest, and why not plain pandas?

---
## 18. Capstone: real data (`reviews.csv`)

This folder has `reviews.csv`, a set of user reviews of a video game, with the columns `platform`, `review_id`, `published_date`, `score` (0–10) and `review_text`. Note that review text contains newlines, which `read_csv` handles because those fields are quoted.

Work through the questions **without looking at the solutions first**. Each one uses techniques from a specific section (shown in brackets).

In [ ]:
reviews = pd.read_csv("reviews.csv", parse_dates=["published_date"])
reviews.info()

### 🧩 Exercise: Capstone part 1: explore

1. [§3] How many reviews are there per platform?
2. [§9] What does the **distribution** of scores look like? Count the reviews for each score from 0 to 10, sorted by score (not by count). What's unusual about it?
3. [§10] Average score per platform, and the number of reviews behind each average.

In [ ]:
# your code here

<details>
<summary><b>Show solution</b> (try first!)</summary>

```python
print(reviews["platform"].value_counts(), end="\n\n")
print(reviews["score"].value_counts().sort_index(), end="\n\n")
# The distribution is U-shaped: lots of 0s and 10s, few in the middle.
# That's typical of user reviews of a polarising release ("review bombing" vs fans).
print(reviews.groupby("platform")["score"].agg(["mean", "count"]))
```
</details>

### 🧩 Exercise: Capstone part 2: derive and compare

1. [§5, §8] Add a `text_length` column (number of characters in `review_text`) and a `sentiment` column: `"negative"` for scores 0–3, `"mixed"` for 4–6, `"positive"` for 7–10 (hint: `pd.cut` or `np.select`).
2. [§10] Do negative or positive reviewers write longer reviews? Compare the median `text_length` per sentiment.
3. [§12] Make a table with platforms as rows, sentiments as columns, and the **percentage** of each platform's reviews in each sentiment (each row should sum to 100).

In [ ]:
# your code here

<details>
<summary><b>Show solution</b> (try first!)</summary>

```python
reviews["text_length"] = reviews["review_text"].str.len()
reviews["sentiment"] = pd.cut(reviews["score"], bins=[-1, 3, 6, 10],
                              labels=["negative", "mixed", "positive"])

print(reviews.groupby("sentiment", observed=True)["text_length"].median(), end="\n\n")

print((pd.crosstab(reviews["platform"], reviews["sentiment"], normalize="index") * 100).round(1))
```
</details>

### 🧩 Exercise: Capstone part 3: over time and text

1. [§13] How many reviews were published per day, and what was the average score each day? (One table, two columns.)
2. [§8, §4] What share of reviews mention "price" or "$" (case-insensitive)? Hint: `.str.contains(r"price|\$", case=False)`.
3. [§4, §9] Show the 3 longest **positive** reviews: only `score`, `text_length` and the first 100 characters of the text (`.str[:100]`).

In [ ]:
# your code here

<details>
<summary><b>Show solution</b> (try first!)</summary>

```python
print(reviews.groupby(reviews["published_date"].dt.date)
             .agg(n_reviews=("review_id", "count"), avg_score=("score", "mean")), end="\n\n")

mentions = reviews["review_text"].str.contains(r"price|\$", case=False)
print(f"Mention price: {mentions.mean():.1%}", end="\n\n")

(reviews[reviews["sentiment"] == "positive"]
    .nlargest(3, "text_length")
    .assign(preview=lambda d: d["review_text"].str[:100])
    [["score", "text_length", "preview"]])
```
</details>

---
## Where to go next

You now have the core of pandas: Series and DataFrames, selection, cleaning, groupby, joins, reshaping, time series, I/O, and an honest picture of when to use something else.

**To lock it in**
* Redo the capstone on a dataset you care about (a Kaggle dataset, your own bank-export CSV, a public API). The mistakes you make on unfamiliar data teach more than any tutorial.
* Try writing each capstone answer as one method chain.

**Next topics**
* **Plotting:** `df.plot()`, then seaborn for statistical charts.
* **MultiIndex** in depth (`xs`, `stack`/`unstack`, `swaplevel`).
* **Window functions:** `expanding`, `ewm`, `groupby().rolling()`.
* **Polars or DuckDB:** re-implement Section 15's chain and compare the speed on a large file.

**References**
* pandas user guide: https://pandas.pydata.org/docs/user_guide/index.html
* *What's new in 3.0* (to understand old tutorials): https://pandas.pydata.org/docs/whatsnew/index.html
* *Python for Data Analysis*, 3rd ed. by Wes McKinney (pandas' creator), free online: https://wesmckinney.com/book/